In [2]:
import os
import requests
import subprocess

#from pyrosm import OSM
import geopandas as gpd
import pandas as pd

from pathlib import Path


In [3]:
## check if osmium is installed

try:
    result = subprocess.run(['osmium', '--version'], check=True, capture_output=True, text=True)
    print(f"Osmium version: {result.stdout.strip()}")
except subprocess.CalledProcessError as e:
    print(f"Error running Osmium: {e}")

Osmium version: osmium version 1.16.0
libosmium version 2.20.0
Supported PBF compression types: none zlib lz4

Copyright (C) 2013-2023  Jochen Topf <jochen@topf.org>
License: GNU GENERAL PUBLIC LICENSE Version 3 <https://gnu.org/licenses/gpl.html>.
This is free software: you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.


what this does:

1. Download OSM data from geofabrik
2. Extract  highways using osmium
3. Convert pbf to geojson.gz and add tags using ogr2ogr
3. Convert geojson.gz to pmtiles using tippecanoe



In [4]:

#### Downloading OSM data from Geofabrik

set_date = "250528" # 2025-05-28


#https://download.geofabrik.de/europe/germany/berlin-250401.osm.pbf
#	germany-250405.osm.pbf

def download_geofabrik_pbf(filename,base_url = "https://download.geofabrik.de/europe/"):
    folder_download = "osm_geofabrik_pbf"
    os.makedirs(folder_download, exist_ok=True)
    
    #filename = "germany-250401.osm.pbf"
    file_path = os.path.join(folder_download, filename)
    file_url = base_url + filename
    
    if os.path.exists(file_path):
        print(f"File already exists: {file_path}, skipping download.")
    else:
        print(f"Downloading: {file_url}")
        response = requests.get(file_url, stream=True, timeout=60)
        if response.status_code == 200:
            with open(file_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=1024):
                    f.write(chunk)
            print(f"Downloaded: {file_path}")
        else:
            print(f"Failed to download {file_url} (Status code: {response.status_code})")


# osmium needs to be installed on your system in order to run this code/filtering
# https://osmcode.org/osmium-tool/
# for my win11 machine i used https://trac.osgeo.org/osgeo4w/

def run_osmium(filename):
    try:
        folder_download = "osm_geofabrik_pbf"
        folder_processed = "processed_osm_files"
        os.makedirs(folder_processed, exist_ok=True)
        
        input_pbf = os.path.join(folder_download, filename)
        filtered_pbf = os.path.join(folder_processed, f"processed_crossings_germany_{set_date}.pbf")

        # # Convert to Unix-style paths using forward slashes
        # input_pbf = input_pbf.replace("\\", "/")
        # filtered_pbf = filtered_pbf.replace("\\", "/")

        if os.path.exists(filtered_pbf):
            print(f"Processed file already exists: {filtered_pbf}, skipping processing.")
            return

        filter_command = [
            "osmium", "tags-filter",
            input_pbf,
            #"w/highway",
            "n/highway=crossing",
            "-o", filtered_pbf
        ]
        print("🔹 Running: ", " ".join(filter_command))
        subprocess.run(filter_command, check=True)

        print("✅ Osmium processing complete! Files saved in 'processed_osm_files/'")

    except subprocess.CalledProcessError as e:
        print("❌ Error running Osmium:", e)




filename = f"germany-{set_date}.osm.pbf"

download_geofabrik_pbf(filename )

run_osmium(filename)


File already exists: osm_geofabrik_pbf/germany-250528.osm.pbf, skipping download.
🔹 Running:  osmium tags-filter osm_geofabrik_pbf/germany-250528.osm.pbf n/highway=crossing -o processed_osm_files/processed_crossings_germany_250528.pbf
✅ Osmium processing complete! Files saved in 'processed_osm_files/'


In [11]:
### CONVERT TO GEOJSON:GZ (ready for tippecanoe)

In [7]:
def ogr2ogr_convert(
    input_file,
    output_file,
    layer="points",
    output_format=None,
    select_fields=None,
    osmconf_path=None
):
    ogr2ogr_path = "ogr2ogr"  # Use system-installed ogr2ogr on WSL

    input_file = Path(input_file).resolve()
    output_file = Path(output_file).resolve()

    cmd = [
        ogr2ogr_path,
        "-f", output_format,
        str(output_file),
        str(input_file),
    ]

    if layer:
        cmd.append(layer)

    if select_fields:
        cmd += ["-select", ",".join(select_fields)]

    env = os.environ.copy()
    if osmconf_path:
        env["OSM_CONFIG_FILE"] = str(Path(osmconf_path).resolve())

    print("Running:", " ".join(cmd))
    try:
        result = subprocess.run(
            cmd,
            check=True,
            capture_output=True,
            text=True,
            env=env
        )
        print("✅ Success")
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("❌ ogr2ogr failed:")
        print("STDERR:", e.stderr)
        print("STDOUT:", e.stdout)
        raise


In [9]:
set_date = "250528"


ogr2ogr_convert(
    input_file = Path(f"processed_osm_files/processed_crossings_germany_{set_date}.pbf"),
    output_file = Path(f"processed_osm_files/processed_crossings_germany_{set_date}.geojson.gz"),
    osmconf_path = Path(f"processed_osm_files/osmconf.ini"),
    output_format="GeoJSON",
    layer="points",
)

Running: ogr2ogr -f GeoJSON /home/simon/unfallkarte/preprocessing/processed_osm_files/processed_crossings_germany_250528.geojson.gz /home/simon/unfallkarte/preprocessing/processed_osm_files/processed_crossings_germany_250528.pbf points
✅ Success
0...10...20...30...40...50...60...70...80...90...100 - done.



In [ ]:
### create PMTILES from GeoJSON.GZ

In [12]:
#def geojson_to_pmtiles(input_geojson, output_pmtiles, layer_name="highways"):
def geojson_to_pmtiles(input_geojson, output_pmtiles, layer_name="crossings", min_zoom=6, max_zoom=14):
    import subprocess
    from pathlib import Path

    input_geojson = str(Path(input_geojson).resolve())
    output_pmtiles = str(Path(output_pmtiles).resolve())

    cmd = [
        "tippecanoe",
        "-o", output_pmtiles,
        "--layer", layer_name,
        "-Z", str(min_zoom),
        "-z", str(max_zoom),
        "--no-feature-limit",
        "--no-tile-size-limit",
        "--drop-densest-as-needed",
        "--drop-rate=0",
        input_geojson
    ]

    print("Running:", " ".join(cmd))
    try:
        subprocess.run(cmd, check=True)
        print("✅ PMTiles generation successful")
    except subprocess.CalledProcessError as e:
        print("❌ Tippecanoe failed:")
        print("STDERR:", e.stderr)
        raise


geojson_to_pmtiles(
    input_geojson="processed_osm_files/processed_crossings_germany_250528.geojson.gz",
    output_pmtiles="processed_osm_files/processed_crossings_germany_250528.pmtiles",
    layer_name="crossings"
)

Running: tippecanoe -o /home/simon/unfallkarte/preprocessing/processed_osm_files/processed_crossings_germany_250528.pmtiles --layer crossings -Z 6 -z 14 --no-feature-limit --no-tile-size-limit --drop-densest-as-needed --drop-rate=0 /home/simon/unfallkarte/preprocessing/processed_osm_files/processed_crossings_germany_250528.geojson.gz


422588 features, 13583545 bytes of geometry and attributes, 5899130 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
  99.9%  14/8712/5670  


✅ PMTiles generation successful


------------------

In [ ]:
### TO PArQUET (only works with special env, dont need it for tippecanoe)

In [4]:
import subprocess
from pathlib import Path
import os

def ogr2ogr_parquet(input_file, output_parquet, layer="lines", select_fields=None, osmconf_path=None):
    ogr2ogr_path = "ogr2ogr"  # Use system-installed ogr2ogr on WSL

    input_file = Path(input_file).resolve()
    output_parquet = Path(output_parquet).resolve()

    cmd = [
        ogr2ogr_path,
        "-f", "Parquet",
        str(output_parquet),
        str(input_file),
    ]

    if layer:
        cmd.append(layer)

    #if select_fields:
    #    cmd += ["-select", ",".join(select_fields)]

    # Prepare environment with optional OSM config path
    env = os.environ.copy()
    if osmconf_path:
        env["OSM_CONFIG_FILE"] = str(Path(osmconf_path).resolve())

    print("Running:", " ".join(cmd))
    try:
        result = subprocess.run(
            cmd,
            check=True,
            capture_output=True,
            text=True,
            env=env
        )
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("❌ ogr2ogr failed:")
        print("STDERR:", e.stderr)
        print("STDOUT:", e.stdout)
        raise


In [ ]:
set_date = "250528"

input_pbf = Path(f"/home/simon/unfallkarte/preprocessing/processed_osm_files/processed_major_highways_germany_{set_date}.pbf")
output_file = Path(f"processed_osm_files/processed_major_highways_germany_{set_date}.parquet")
osmconf_file = Path(f"/home/simon/unfallkarte/preprocessing/processed_osm_files/osmconf.ini")

#selected_fields = ["maxspeed", "name", "highway"]

ogr2ogr_parquet(
    input_pbf,
    output_file,
   # select_fields=selected_fields,
    osmconf_path=osmconf_file
)

Running: ogr2ogr -f Parquet /home/simon/unfallkarte/preprocessing/processed_osm_files/processed_major_highways_germany_250528_otherenv2.parquet /home/simon/unfallkarte/preprocessing/processed_osm_files/processed_major_highways_germany_250528.pbf lines
0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:10.

